# Request Anatomy: System, User, Assistant

**Phase 01** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-01/01-request-anatomy.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 01-01: Request Anatomy - System, User, Assistant
Phase 01: Prompt and Context Engineering

Demonstrates the three-role conversation structure of the Anthropic Messages API.
Every concept in the lesson is executable here.
"""

import anthropic

client = anthropic.Anthropic()

### Demo 1: Single-turn request - raw dict construction

In [ ]:
def demo_single_turn() -> None:
    """A single-turn request built as plain Python dicts."""
    print("=" * 60)
    print("DEMO 1: Single-turn request")
    print("=" * 60)

    messages = [
        {"role": "user", "content": "What is the capital of France?"}
    ]

    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=128,
        system="You are a geography assistant. Answer in one sentence.",
        messages=messages
    )

    print(f"Response: {response.content[0].text}")
    print(f"Stop reason: {response.stop_reason}")
    print(f"Tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out")

### Demo 2: Multi-turn request - manually built conversation history

In [ ]:
def demo_multi_turn() -> None:
    """Manually build a 3-turn conversation and send it as a single API call."""
    print("\n" + "=" * 60)
    print("DEMO 2: Multi-turn request (manual history)")
    print("=" * 60)

    # The messages array is just a list. The model reads it top to bottom.
    messages = [
        {"role": "user",      "content": "My name is Alex. Remember it."},
        {"role": "assistant", "content": "Got it, Alex. How can I help you?"},
        {"role": "user",      "content": "What is my name?"},
    ]

    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=128,
        system="You are a helpful assistant.",
        messages=messages
    )

    print(f"Response: {response.content[0].text}")
    print("\nNow remove the first two turns and ask the same question...")

    # Remove the context - the model will not know the name
    messages_no_context = [
        {"role": "user", "content": "What is my name?"},
    ]

    response2 = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=128,
        system="You are a helpful assistant.",
        messages=messages_no_context
    )

    print(f"Response (no context): {response2.content[0].text}")
    print("=> Context removed = information lost. There is no memory between calls.")

### Demo 3: Role validation - catching structural errors

In [ ]:
def validate_messages(messages: list) -> list[str]:
    """
    Validate a messages array for structural correctness.
    Returns a list of error strings (empty list = valid).
    """
    errors = []

    if not messages:
        errors.append("messages array is empty")
        return errors

    if messages[0]["role"] != "user":
        errors.append(f"First message must be 'user', got '{messages[0]['role']}'")

    for i in range(1, len(messages)):
        prev_role = messages[i - 1]["role"]
        curr_role = messages[i]["role"]
        if prev_role == curr_role:
            errors.append(
                f"Turn {i}: consecutive '{curr_role}' messages not allowed "
                f"(index {i-1} and {i})"
            )

    return errors


def demo_validation() -> None:
    """Show role validation catching structural problems."""
    print("\n" + "=" * 60)
    print("DEMO 3: Role structure validation")
    print("=" * 60)

    valid_messages = [
        {"role": "user",      "content": "Hello"},
        {"role": "assistant", "content": "Hi there"},
        {"role": "user",      "content": "How are you?"},
    ]
    errors = validate_messages(valid_messages)
    print(f"Valid messages: {errors or 'OK'}")

    invalid_messages = [
        {"role": "user",  "content": "First message"},
        {"role": "user",  "content": "Second message"},   # two user turns in a row
    ]
    errors = validate_messages(invalid_messages)
    print(f"Invalid messages (double user): {errors}")

    starts_with_assistant = [
        {"role": "assistant", "content": "I speak first"},
        {"role": "user",      "content": "OK"},
    ]
    errors = validate_messages(starts_with_assistant)
    print(f"Invalid messages (starts with assistant): {errors}")

### Demo 4: Interactive chat loop - shows manual turn management

In [ ]:
def chat(system_prompt: str = "You are a helpful assistant.") -> None:
    """
    Simple REPL demonstrating manual turn management.
    The messages list grows by 2 each turn (user + assistant).
    """
    messages = []
    print("\n" + "=" * 60)
    print("DEMO 4: Chat loop with manual history")
    print("Type 'quit' to exit. Watch the input token count grow each turn.")
    print("=" * 60 + "\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            break
        if not user_input:
            continue

        messages.append({"role": "user", "content": user_input})

        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=512,
            system=system_prompt,
            messages=messages
        )

        assistant_text = response.content[0].text

        # Append the assistant response - required for coherent next turn
        messages.append({"role": "assistant", "content": assistant_text})

        print(f"\nClaude: {assistant_text}")
        print(f"[{len(messages)} turns | {response.usage.input_tokens} input tokens "
              f"| {response.usage.output_tokens} output tokens]\n")

### Demo 5: System prompt isolation - same question, 3 different behaviors

In [ ]:
def demo_system_prompt_impact() -> None:
    """Show that the system parameter fundamentally changes model behavior."""
    print("\n" + "=" * 60)
    print("DEMO 5: System prompt isolation")
    print("=" * 60)

    question = "What is machine learning?"

    system_prompts = [
        ("Formal",     "You are a technical writer. Use formal academic language."),
        ("One word",   "You are a minimalist. Respond in exactly one word."),
        ("JSON",       'You respond only in JSON format: {"answer": "..."}'),
    ]

    for label, system in system_prompts:
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=128,
            system=system,
            messages=[{"role": "user", "content": question}]
        )
        print(f"\n[{label}] {response.content[0].text[:120]}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

### Demo

In [ ]:
demo_single_turn()
demo_multi_turn()
demo_validation()
demo_system_prompt_impact()

print("\n" + "=" * 60)
print("DEMO 4: Interactive chat (optional)")
print("=" * 60)
run_chat = input("Run interactive chat demo? (y/n): ").strip().lower()
if run_chat == "y":
    chat("You are a helpful assistant. Be concise.")

---

## Self-Check

Answer these without running code first:

**1. What is the most likely cause?**

- A. The model's context window has been exceeded and it lost turn 2.
- B. The model has a bug where it forgets information after N turns.
- C. Your conversation management code is truncating old turns to reduce token costs, and turn 2 is being dropped.
- D. The system prompt is overriding the user's information.

**2. What is the practical difference between putting this instruction in the user message vs. the system parameter?**

- A. No difference. The model treats all instructions equally regardless of which role they come from.
- B. The system parameter is read first and sets the frame for all subsequent messages. A user-message instruction is more likely to be overridden by subsequent user turns that don't mention JSON.
- C. User message instructions are always stronger than system instructions.
- D. The system parameter only works for single-turn requests.

**3. What is wrong and how do you fix it?**

- A. The messages array cannot have more than one entry.
- B. The 'Hello' message is too short and triggers a validation error.
- C. Two consecutive user messages violate the alternating role requirement. Add an assistant message between them or merge the two user messages into one.
- D. The content strings need to be wrapped in content block objects, not plain strings.

**4. What additional information do you need to fully reproduce the bug?**

- A. The model version and temperature setting.
- B. The complete messages array (all turns) and the system parameter that were sent in the failing API call.
- C. The user's account ID and session token.
- D. The model's internal reasoning chain.

**5. What specific production risk does this attitude create?**

- A. None. Frameworks handle conversation management correctly by design.
- B. The framework may truncate, reorder, or modify messages in ways that cause subtle bugs. Without understanding the messages array, you cannot debug unexpected model behavior.
- C. Frameworks always send more tokens than necessary, increasing API costs.
- D. The model behaves differently when called through a framework.

**6. What effect does this have?**

- A. It causes an API error because the last message must always be from the user.
- B. The model ignores the prefilled assistant content and generates a fresh response.
- C. The model treats the prefilled assistant content as its own previous output and continues from where it left off.
- D. The model will repeat the prefilled content verbatim and then add new content.

**7. What does this confirm about how the API works?**

- A. The model is caching previous turns and only processing new content.
- B. The model sends only the last 2-3 turns to save bandwidth.
- C. The entire messages array is sent and processed on every API call. Input tokens grow with conversation length because the full history is included each time.
- D. Token count growth indicates a memory leak in the SDK.

**8. What is happening and what is the correct fix?**

- A. The model encountered an error and stopped early. Retry the request.
- B. The model's context window was exceeded. Reduce the size of the messages array.
- C. The output reached the max_tokens limit you set. Increase max_tokens to allow a longer response.
- D. The system prompt is conflicting with the user message. Remove the system prompt.

_Answers are in `checks.json` in the lesson directory._